# Caderno 7.3 - Pesquisa em dois estágios [runs]

## 1. Parâmetros

In [1]:
PASTA_DOCS_QRELS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/'
PASTA_RESULTADO_CADERNO = './dados/outputs/7 - runs/'

ARQUIVO_QRELS = f'{PASTA_DOCS_QRELS}/qrel.csv'

## 2. IR de dois estágios

In [2]:
import pandas as pd

qrels = pd.read_csv(ARQUIVO_QRELS)

In [3]:
def dois_estagios(run1, run2, top_n=300, nome_arquivo=None):
    # ============================================================
    # 1. Carregar runs
    # ============================================================
    first_stage = pd.read_csv(run1)
    second_stage = pd.read_csv(run2)
    
    # Garantir tipos corretos
    first_stage["RANK"] = first_stage["RANK"].astype(int)
    second_stage["RANK"] = second_stage["RANK"].astype(int)
    
    # ============================================================
    # 2. Pegar top-n do primeiro estágio
    # ============================================================
    first_stage_top_n = (
        first_stage
        .sort_values(["QUERY_KEY", "RANK"])
        .groupby("QUERY_KEY")
        .head(top_n)
    )
    
    # ============================================================
    # 3. Juntar com o segundo estágio
    # ============================================================
    # Renomear coluna para evitar conflito
    second_stage = second_stage.rename(columns={"RANK": "SECOND_STAGE_RANK"})
    
    # Merge mantendo apenas docs que estão no top-300 do primeiro estágio
    reranked = first_stage_top_n.merge(
        second_stage[["QUERY_KEY", "DOC_KEY", "SECOND_STAGE_RANK"]],
        on=["QUERY_KEY", "DOC_KEY"],
        how="left"
    )
    
    # ============================================================
    # 4. Tratar documentos que não aparecem no run semântico
    # ============================================================
    # Se algum doc não tiver rank semântico, colocamos no final, na ordem que retorna pelo primeiro estágio
    # Guardar o rank original do primeiro estágio
    reranked = reranked.rename(columns={"RANK": "FIRST_STAGE_RANK"})
    # Criar coluna auxiliar indicando se veio do segundo estágio
    reranked["HAS_SECOND_STAGE"] = reranked["SECOND_STAGE_RANK"].notna()
    
    # ============================================================
    # 5. Reordenar pelo rank semântico
    # ============================================================
    # Agora ordenar:
    # 1) Primeiro os que têm segundo estágio
    # 2) Dentro deles, pelo SECOND_STAGE_RANK
    # 3) Depois os que não têm, mantendo ordem do FIRST_STAGE_RANK
    reranked = (
        reranked
        .sort_values(
            by=["QUERY_KEY", "HAS_SECOND_STAGE", "SECOND_STAGE_RANK", "FIRST_STAGE_RANK"],
            ascending=[True, False, True, True]
        )
    )
    
    # Criar novo ranking final (1, 2, 3, ...)
    reranked["RANK"] = (
        reranked
        .groupby("QUERY_KEY")
        .cumcount() + 1
    )
    
    # Manter apenas colunas padrão de run
    final_run = reranked[["QUERY_KEY", "DOC_KEY", "RANK"]]

    if nome_arquivo is not None:
        final_run.to_csv(nome_arquivo, index=False)
        
    return final_run

## 3. Pesquisa no IR de dois estágios

In [4]:
from metricas import metricas

primeiro_estagio = f"{PASTA_RESULTADO_CADERNO}run_bm25_1_0.9_texto.csv"
segundo_estagio = f"{PASTA_RESULTADO_CADERNO}run_text-embedding-3-small.csv"
nome_arquivo = f"{PASTA_RESULTADO_CADERNO}run_bm25_1_0.9_texto_text-embedding-3-small.csv"

print(f"Experimento: {nome_arquivo}")
df_resultado = dois_estagios(primeiro_estagio, segundo_estagio, 50, nome_arquivo)

df_metricas = metricas(df_resultado, qrels, aproximacao_trec_eval=True, k=[10])
print(f'\tnDCG@10: {df_metricas['nDCG@10'].mean():.4f}')

Experimento: ./dados/outputs/7 - runs/run_bm25_1_0.9_texto_text-embedding-3-small.csv
	nDCG@10: 0.3064


In [5]:
primeiro_estagio = f"{PASTA_RESULTADO_CADERNO}run_bm25_1_0.9_texto.csv"
segundo_estagio = f"{PASTA_RESULTADO_CADERNO}run_gemini-embedding-001.csv"
nome_arquivo = f"{PASTA_RESULTADO_CADERNO}run_bm25_1_0.9_texto_gemini-embedding-001.csv"

print(f"Experimento: {nome_arquivo}")
df_resultado = dois_estagios(primeiro_estagio, segundo_estagio, 50, nome_arquivo)

df_metricas = metricas(df_resultado, qrels, aproximacao_trec_eval=True, k=[10])
print(f'\tnDCG@10: {df_metricas['nDCG@10'].mean():.4f}')

Experimento: ./dados/outputs/7 - runs/run_bm25_1_0.9_texto_gemini-embedding-001.csv
	nDCG@10: 0.3345
